In [ ]:
%load_ext autoreload
%autoreload 2

from math import pi
from glob import glob

import numpy as np
import matplotlib.pyplot as plt

import ast

import pandas as pd

In [ ]:
from matplotlib.colors import LogNorm
plt.rcParams.update({'font.size': 18})

In [ ]:
from numpy.random import normal

In [ ]:
from scipy.optimize import curve_fit

In [ ]:
f_scatter    = open('xsec_scatter.txt','r')
f_excitation = open('xsec_excitation.txt','r')
f_ioni       = open('xsec_ioni.txt','r')

In [ ]:
def SaveXSEC(file):
    xsec_e = []
    xsec_x = []
    CTR = 0
    file.seek(0)
    for line in file:
        words = line.split()
        if (CTR != 0):
            e = float(words[0])
            x = float(words[1])
            xsec_e.append(e)
            xsec_x.append(x)
        CTR += 1
    #print(xsec_e_scatter)
    #print(xsec_x_scatter)
    return xsec_e,xsec_x

In [ ]:
xsec_e_scatter, xsec_x_scatter = SaveXSEC(f_scatter)
xsec_e_excitation, xsec_x_excitation = SaveXSEC(f_excitation)
xsec_e_ioni, xsec_x_ioni = SaveXSEC(f_ioni)

In [ ]:
fig = plt.figure(figsize=(6,6))
plt.plot(xsec_e_scatter,xsec_x_scatter,'ro-',label='scatter')
plt.plot(xsec_e_excitation,xsec_x_excitation,'bo-',label='excitation')
plt.plot(xsec_e_ioni,xsec_x_ioni,'go-',label='ionization')
plt.xlabel('Energy [eV]')
plt.ylabel(r'$\sigma$   $[m^2]$')
plt.legend(loc=7,fontsize=14)
plt.show()

In [ ]:
def GetXSEC(energy_v,xsec_v,energy):
    emin = min(energy_v)
    emax = max(energy_v)
    # are we out of bounds?
    if (energy < emin): return 0
    if (energy > emax): return 0
    # find which "bin" we belong to
    for i,e in enumerate(energy_v):
        ebin = energy_v[i]
        if (ebin >= energy):
            return xsec_v[i]
    print('ERROR: DID NOT FIND SOLUTION.')
    return 0

In [ ]:
def GetTOTXSEC(energy_v_v,xsec_v_v,energy):
    xsectot = 0
    # check lengths agree
    if (len(energy_v_v) != len(xsec_v_v)):
        return 0
    N = len(energy_v_v)
    for i in range(N):
        xsectot += GetXSEC(energy_v_v[i],xsec_v_v[i],energy)
    return xsectot

In [ ]:
print (GetXSEC(xsec_e_scatter,xsec_x_scatter,10.))

xsec_e_v_v = [xsec_e_scatter,xsec_e_excitation,xsec_e_ioni]
xsec_x_v_v = [xsec_x_scatter,xsec_x_excitation,xsec_x_ioni]
print (GetTOTXSEC(xsec_e_v_v,xsec_x_v_v,10.))

In [ ]:
def GetLambda(xsec,density):
    return 1./(np.sqrt(2)*xsec*density)

In [ ]:
### Let's simulate a particle moving around: what will happen to it?

In [ ]:
p_energy = np.random.random()*40

rho = 10. # 10 particles per meter-square

vel = 3 # meters/second: speed of particle

In [ ]:
# what's the total cross-section at this point?
xsec = GetTOTXSEC(xsec_e_v_v,xsec_x_v_v,p_energy)
print('total cross-section at energy of %g eV is : %g m^2'%(p_energy,xsec))

In [ ]:
# what's the mean-free path?
mfp = GetLambda(xsec,rho)
print('mean free path for a particle of %g eV is : %g meters'%(p_energy,mfp))

In [ ]:
K = 2.0 * xsec*vel # fine-grained scaling factor

In [ ]:
# what is the timestep size:
tau = (K)/(rho*2)
print('the timescale for collisions at %f energy is %g sec.'%(p_energy,tau))

In [ ]:
# simulate time-step
t_step = np.random.exponential(tau)
if (t_step > 3*tau):
    t_step = 3*tau
print('we are simulating a time-step of %g sec.'%t_step)

In [ ]:
# move to the new location based on your velocity vector

In [ ]:
# now figure out what interaction to undergo...
# for each interaction mode, calculate the probability that it occurs at this energy
p_ioni       = (vel * GetXSEC(xsec_e_ioni,xsec_x_ioni,p_energy)) / K
p_scatter    = (vel * GetXSEC(xsec_e_scatter,xsec_x_scatter,p_energy)) / K
p_excitation = (vel * GetXSEC(xsec_e_excitation,xsec_x_excitation,p_energy)) / K
print('probabilities are...')
print('p scatter   : %.02f'%p_scatter)
print('p excitation: %.02f'%p_excitation)
print('p ionization: %.02f'%p_ioni)